In [7]:
import pandas as pd
import regex as re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/siavashj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
df_orig = pd.read_parquet('FNDDSeverything.parquet.gzip')
df = df_orig.select_dtypes(include="str")

In [ ]:

def preprocess_food(text, Tokenize = True):
    """
    Complete preprocessing for food descriptions
    - Removes measurements and numbers
    - Tokenizes
    - Removes stopwords
    - Lemmatizes
    """

    sentence = ''
    
    # Step 0: replace ',' with ' '.
    text = re.sub(r',', ' ', text)
    
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove all measurement patterns
    patterns_to_remove = [
        r'\d+\.?\d*\s?(?:oz|lb|lbs|g|kg|ml|l|cup|cups|tsp|tbsp|pt|qt|gal|mg|mcg|cal|count|ct|pack|pk|piece|pieces|pc|ounce|ounces|pound|pounds|gram|grams|kilogram|kilograms|milliliter|milliliters|liter|liters|teaspoon|teaspoons|tablespoon|tablespoons|pint|pints|quart|quarts|gallon|gallons|milligram|milligrams|microgram|micrograms|calorie|calories|fl|fluid)\b',
        r'\b\d+\.?\d*\b',  # standalone numbers
        r'\d+\.?\d*\s?%',  # percentages
        r'\d+\s?x\s?\d+',  # dimensions
    ]
    
    for pattern in patterns_to_remove:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Step 3: Clean special characters but keep spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 4: Clean multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 5: Tokenize
    tokens = word_tokenize(text)
    
    # Step 6: Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    
    # Step 7: Remove very short tokens
    tokens = [t for t in tokens if len(t) > 2]
    
    # Step 8: Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    if Tokenize:
        return tokens
    else:
        
        sentence = " ".join(tokens)
        return sentence

# Test

for sample in df.description[[24,81,124,134,550,551,552,553]]:
    tokens = preprocess_food(sample, True )
    print(f"Original: {sample}")
    print(f"Tokens:   {tokens}\n")
    sent = preprocess_food(sample, False )
    print(f"after token sent:   {sent}\n")



# def preprocess_column (List = list):
#     L = pd.DataFrame([])
#     for w in List:
#         preprocess_food_entries(w)
#     return L


I have a dataframe of 500k food descriptions. I want to vectorize it and feed the vectors to a neural network for classification. I need help with following programing tasks. 1) I want to use packages like sklearn, word2vector, etc to vectorize it. Suggest different methods to do so in a jupyter notebook code style with mark down pieces that explains about the methods pros and cons. 2) Suggest different neural networks that can do clustring on this dataframe for the aim of finding better classification of the food categories. 3) Do not define functions for doing the task. I want simple codes that I can use for different segments. 4) Make it memory and computationally efficient. If there are functions and features that we can use to adjust computation resources also explain them. 5) Include good visuallization codes so that we can present the result in a presentation. What if we also include the nutrition information of the foods to better classify/categorize this dataframe. 6) Assume that I have the nutrition information. I have already cleaned and tokenized the data using a function

# ============================================================================
# SETUP AND IMPORTS
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA, TruncatedSVD, IncrementalPCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cluster import MiniBatchKMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import umap
import warnings
warnings.filterwarnings('ignore')

# For deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# For Word2Vec and similar
from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# For visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Memory optimization
import gc

print("All packages imported successfully!")

1. Data Vectorization Methods
Method 1: TF-IDF Vectorization

Pros: Fast, interpretable, works well for short texts, memory efficient
Cons: Doesn't capture semantic meaning, produces sparse matrices, ignores word order

# ============================================================================
# METHOD 1: TF-IDF VECTORIZATION
# ============================================================================

def tfidf_vectorization(texts, max_features=5000, ngram_range=(1, 2), batch_size=10000):
    """
    TF-IDF vectorization with memory optimization
    
    Parameters:
    -----------
    texts: array-like of strings
    max_features: maximum number of features to extract
    ngram_range: tuple, range of n-grams
    batch_size: for memory management
    """
    print("Starting TF-IDF Vectorization...")
    
    # Initialize vectorizer
    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        min_df=5,  # ignore terms appearing in less than 5 documents
        max_df=0.8,  # ignore terms appearing in more than 80% of documents
        sublinear_tf=True,  # apply sublinear tf scaling
        strip_accents='unicode',
        lowercase=True
    )
    
    # Fit and transform
    vectors = tfidf.fit_transform(texts)
    
    print(f"TF-IDF Shape: {vectors.shape}")
    print(f"Sparsity: {(1.0 - vectors.nnz / (vectors.shape[0] * vectors.shape[1])) * 100:.2f}%")
    
    return vectors, tfidf

# Example usage (assuming df is your dataframe with 'description' column)
# Assuming you've already applied food_clean function
# df['cleaned_text'] = df['description'].apply(food_clean)

# For demonstration
sample_texts = [
    "grilled chicken breast with vegetables",
    "chocolate chip cookies with milk",
    "fresh garden salad with olive oil"
] * 1000  # Simulating data

tfidf_vectors, tfidf_model = tfidf_vectorization(sample_texts, max_features=3000)

# Get feature names for interpretation
feature_names = tfidf_model.get_feature_names_out()
print(f"\nSample features: {feature_names[:10]}")

Method 2: Word2Vec with Averaging

Pros: Captures semantic meaning, produces dense vectors
Cons: Simple averaging loses sequence information, requires training or pre-trained models

# ============================================================================
# METHOD 2: WORD2VEC WITH AVERAGING
# ============================================================================

def word2vec_vectorization(tokenized_texts, vector_size=100, window=5, min_count=5, workers=4):
    """
    Train Word2Vec and create document vectors by averaging word vectors
    
    Parameters:
    -----------
    tokenized_texts: list of lists of tokens
    vector_size: dimensionality of word vectors
    window: context window size
    min_count: minimum word frequency
    """
    print("Training Word2Vec model...")
    
    # Train Word2Vec model
    w2v_model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=1,  # Skip-gram (1) or CBOW (0)
        epochs=10,
        seed=42
    )
    
    print(f"Vocabulary size: {len(w2v_model.wv)}")
    
    # Create document vectors by averaging word vectors
    def document_vector(tokens):
        vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
        if len(vectors) == 0:
            return np.zeros(vector_size)
        return np.mean(vectors, axis=0)
    
    print("Creating document vectors...")
    doc_vectors = np.array([document_vector(tokens) for tokens in tokenized_texts])
    
    print(f"Document vectors shape: {doc_vectors.shape}")
    
    return doc_vectors, w2v_model

# Example usage
# Assuming cleaned texts are tokenized
def simple_tokenize(text):
    return text.lower().split()

tokenized_sample = [simple_tokenize(text) for text in sample_texts]
w2v_vectors, w2v_model = word2vec_vectorization(tokenized_sample, vector_size=100)

# Find similar words (useful for exploration)
if len(w2v_model.wv) > 0:
    sample_word = list(w2v_model.wv.index_to_key)[0]
    print(f"\nWords similar to '{sample_word}':")
    try:
        print(w2v_model.wv.most_similar(sample_word, topn=5))
    except:
        print("Not enough vocabulary to show similarities")

Method 3: Doc2Vec

Pros: Learns document-level representations, considers context
Cons: Slower to train, requires more data for good performance

# ============================================================================
# METHOD 3: DOC2VEC
# ============================================================================

def doc2vec_vectorization(tokenized_texts, vector_size=100, epochs=20, min_count=5):
    """
    Train Doc2Vec model for document embeddings
    
    Parameters:
    -----------
    tokenized_texts: list of lists of tokens
    vector_size: dimensionality of document vectors
    epochs: number of training epochs
    """
    print("Preparing tagged documents for Doc2Vec...")
    
    # Tag documents
    tagged_docs = [TaggedDocument(words=tokens, tags=[str(i)]) 
                   for i, tokens in enumerate(tokenized_texts)]
    
    print("Training Doc2Vec model...")
    
    # Train Doc2Vec
    d2v_model = Doc2Vec(
        documents=tagged_docs,
        vector_size=vector_size,
        window=5,
        min_count=min_count,
        workers=4,
        epochs=epochs,
        dm=1,  # Distributed Memory (1) or Distributed BOW (0)
        seed=42
    )
    
    print(f"Vocabulary size: {len(d2v_model.wv)}")
    
    # Get document vectors
    doc_vectors = np.array([d2v_model.dv[str(i)] for i in range(len(tokenized_texts))])
    
    print(f"Document vectors shape: {doc_vectors.shape}")
    
    return doc_vectors, d2v_model

# Example usage
d2v_vectors, d2v_model = doc2vec_vectorization(tokenized_sample, vector_size=100, epochs=10)

Method 4: FastText (Better for morphologically rich text)

Pros: Handles out-of-vocabulary words, captures subword information
Cons: Slightly slower than Word2Vec, larger model size

# ============================================================================
# METHOD 4: FASTTEXT
# ============================================================================

def fasttext_vectorization(tokenized_texts, vector_size=100, window=5, min_count=5):
    """
    Train FastText model and create document vectors
    
    Parameters:
    -----------
    tokenized_texts: list of lists of tokens
    vector_size: dimensionality of word vectors
    """
    print("Training FastText model...")
    
    # Train FastText model
    ft_model = FastText(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        sg=1,  # Skip-gram
        epochs=10,
        seed=42
    )
    
    print(f"Vocabulary size: {len(ft_model.wv)}")
    
    # Create document vectors by averaging
    def document_vector(tokens):
        vectors = [ft_model.wv[word] for word in tokens if word in ft_model.wv]
        if len(vectors) == 0:
            return np.zeros(vector_size)
        return np.mean(vectors, axis=0)
    
    print("Creating document vectors...")
    doc_vectors = np.array([document_vector(tokens) for tokens in tokenized_texts])
    
    print(f"Document vectors shape: {doc_vectors.shape}")
    
    return doc_vectors, ft_model

# Example usage
ft_vectors, ft_model = fasttext_vectorization(tokenized_sample, vector_size=100)

Method 5: Sentence Transformers (BERT-based) - Most Powerful

Pros: State-of-the-art semantic understanding, pre-trained models available
Cons: More computationally expensive, requires more memory

# ============================================================================
# METHOD 5: SENTENCE TRANSFORMERS (BERT-based)
# ============================================================================

from sentence_transformers import SentenceTransformer

def bert_vectorization(texts, model_name='all-MiniLM-L6-v2', batch_size=32):
    """
    Use pre-trained BERT model for semantic embeddings
    
    Parameters:
    -----------
    texts: list of text strings
    model_name: name of pre-trained model
    batch_size: batch size for encoding
    
    Popular models:
    - 'all-MiniLM-L6-v2': Fast, good performance (384 dim)
    - 'all-mpnet-base-v2': Best quality (768 dim)
    - 'paraphrase-MiniLM-L6-v2': Good for paraphrase detection
    """
    print(f"Loading {model_name} model...")
    
    # Load pre-trained model
    model = SentenceTransformer(model_name)
    
    print("Encoding texts...")
    # Encode in batches to manage memory
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    print(f"Embeddings shape: {embeddings.shape}")
    
    return embeddings, model

# Example usage (commented out to save memory in demo)
# bert_vectors, bert_model = bert_vectorization(sample_texts[:1000], batch_size=32)

2. Dimensionality Reduction (For Visualization & Memory Efficiency)

# ============================================================================
# DIMENSIONALITY REDUCTION
# ============================================================================

def reduce_dimensions(vectors, method='pca', n_components=50, sparse_input=False):
    """
    Reduce dimensionality of vectors
    
    Parameters:
    -----------
    vectors: input vectors (sparse or dense)
    method: 'pca', 'svd', 'umap', 'tsne'
    n_components: number of components to keep
    sparse_input: whether input is sparse matrix
    """
    print(f"Reducing dimensions using {method.upper()}...")
    
    if method == 'pca':
        if sparse_input:
            # Use TruncatedSVD for sparse matrices
            reducer = TruncatedSVD(n_components=n_components, random_state=42)
        else:
            reducer = PCA(n_components=n_components, random_state=42)
        reduced = reducer.fit_transform(vectors)
        explained_var = sum(reducer.explained_variance_ratio_)
        print(f"Explained variance: {explained_var:.2%}")
        
    elif method == 'svd':
        reducer = TruncatedSVD(n_components=n_components, random_state=42)
        reduced = reducer.fit_transform(vectors)
        explained_var = sum(reducer.explained_variance_ratio_)
        print(f"Explained variance: {explained_var:.2%}")
        
    elif method == 'umap':
        reducer = umap.UMAP(
            n_components=n_components,
            n_neighbors=15,
            min_dist=0.1,
            metric='cosine',
            random_state=42
        )
        reduced = reducer.fit_transform(vectors)
        
    elif method == 'tsne':
        # t-SNE is memory intensive, use PCA first if needed
        if vectors.shape[1] > 50:
            print("Pre-reducing with PCA before t-SNE...")
            pca = PCA(n_components=50, random_state=42)
            vectors = pca.fit_transform(vectors)
        
        reducer = TSNE(
            n_components=n_components,
            perplexity=30,
            random_state=42,
            n_jobs=-1
        )
        reduced = reducer.fit_transform(vectors)
    
    print(f"Reduced shape: {reduced.shape}")
    return reduced, reducer

# Example usage
# For TF-IDF (sparse)
tfidf_reduced, tfidf_reducer = reduce_dimensions(
    tfidf_vectors, 
    method='svd', 
    n_components=100, 
    sparse_input=True
)

# For Word2Vec (dense)
w2v_reduced, w2v_reducer = reduce_dimensions(
    w2v_vectors, 
    method='pca', 
    n_components=50
)

# For 2D visualization
tfidf_2d, _ = reduce_dimensions(tfidf_reduced, method='umap', n_components=2)

3. Neural Network Architectures for Clustering
Architecture 1: Autoencoder for Unsupervised Feature Learning

Pros: Learns compressed representations, good for clustering
Cons: Requires tuning, can be slow to train

# ============================================================================
# AUTOENCODER FOR DIMENSIONALITY REDUCTION AND CLUSTERING
# ============================================================================

def build_autoencoder(input_dim, encoding_dims=[256, 128, 64], latent_dim=32):
    """
    Build autoencoder for unsupervised feature learning
    
    Parameters:
    -----------
    input_dim: dimension of input vectors
    encoding_dims: list of hidden layer dimensions
    latent_dim: dimension of latent representation
    """
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,))
    x = encoder_input
    
    for dim in encoding_dims:
        x = layers.Dense(dim, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
    
    # Latent representation
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)
    
    encoder = Model


Food Description Vectorization and Neural Network Classification
Dataset Preparation

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your dataframe is loaded
# df = pd.read_csv('food_descriptions.csv')
# For demonstration, let's assume you have:
# df with columns: 'description' and 'category'

# Basic preprocessing
df['description_clean'] = df['description'].str.lower().str.strip()
df['description_clean'] = df['description_clean'].str.replace('[^\w\s]', '', regex=True)

Method 1: Count Vectorizer (Bag of Words)

Pros:

    Simple and intuitive
    Fast to compute
    Good baseline for text classification
    Preserves exact word frequencies

Cons:

    Ignores word order and context
    Creates very high-dimensional sparse matrices
    No semantic understanding
    Common words dominate

# Initialize Count Vectorizer
count_vectorizer = CountVectorizer(
    max_features=5000,  # Limit vocabulary size
    max_df=0.8,         # Ignore terms appearing in >80% of documents
    min_df=5,           # Ignore terms appearing in <5 documents
    ngram_range=(1, 2)  # Unigrams and bigrams
)

# Fit and transform
X_count = count_vectorizer.fit_transform(df['description_clean'])

# Convert to dense array if needed (careful with memory!)
# X_count_dense = X_count.toarray()

print(f"Count Vectorizer Shape: {X_count.shape}")
print(f"Vocabulary Size: {len(count_vectorizer.vocabulary_)}")

# View feature names
feature_names_count = count_vectorizer.get_feature_names_out()
print(f"Sample features: {feature_names_count[:20]}")

Method 2: Binary Vectorizer

Pros:

    Reduces impact of word repetition
    Smaller memory footprint than count
    Good for short texts
    Less affected by document length

Cons:

    Loses frequency information
    Still ignores context
    High dimensionality
    No semantic relationships

# Initialize Binary Vectorizer
binary_vectorizer = CountVectorizer(
    max_features=5000,
    max_df=0.8,
    min_df=5,
    binary=True,           # Key difference: binary encoding
    ngram_range=(1, 2)
)

# Fit and transform
X_binary = binary_vectorizer.fit_transform(df['description_clean'])

print(f"Binary Vectorizer Shape: {X_binary.shape}")
print(f"Non-zero elements: {X_binary.nnz}")

# Compare with count vectorizer
print(f"Sparsity: {(1 - X_binary.nnz / (X_binary.shape[0] * X_binary.shape[1])) * 100:.2f}%")

Method 3: TF-IDF Vectorizer

Pros:

    Reduces importance of common words
    Better than raw counts for most tasks
    Highlights distinctive terms
    Still relatively fast

Cons:

    Still no semantic understanding
    Doesn't capture word order
    Can be sensitive to parameter tuning
    Assumes term independence

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    max_df=0.8,
    min_df=5,
    ngram_range=(1, 2),
    sublinear_tf=True,    # Use log scaling for term frequency
    use_idf=True,
    smooth_idf=True
)

# Fit and transform
X_tfidf = tfidf_vectorizer.fit_transform(df['description_clean'])

print(f"TF-IDF Shape: {X_tfidf.shape}")

# Get top terms by average TF-IDF score
tfidf_scores = np.array(X_tfidf.mean(axis=0)).flatten()
top_indices = tfidf_scores.argsort()[-20:][::-1]
feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()
print("Top 20 terms by TF-IDF:")
for idx in top_indices:
    print(f"{feature_names_tfidf[idx]}: {tfidf_scores[idx]:.4f}")

Method 4: Word2Vec (Average Word Embeddings)

Pros:

    Captures semantic relationships
    Dense representations (lower dimension)
    Pre-trained models available
    Words with similar meaning have similar vectors

Cons:

    Requires more computational resources
    Needs more data to train well
    Averaging embeddings loses sentence structure
    May miss domain-specific terminology

# Tokenize sentences for Word2Vec
tokenized_descriptions = [desc.split() for desc in df['description_clean']]

# Train Word2Vec model
w2v_model = Word2Vec(
    sentences=tokenized_descriptions,
    vector_size=100,      # Embedding dimension
    window=5,             # Context window size
    min_count=5,          # Minimum word frequency
    workers=4,            # Parallel processing
    sg=0,                 # 0=CBOW, 1=Skip-gram
    epochs=10
)

print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

# Create document vectors by averaging word vectors
X_w2v = np.zeros((len(df), 100))

for i, tokens in enumerate(tokenized_descriptions):
    valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if valid_vectors:
        X_w2v[i] = np.mean(valid_vectors, axis=0)
    else:
        X_w2v[i] = np.zeros(100)

print(f"Word2Vec vectors shape: {X_w2v.shape}")

# Find similar words (example)
if 'chicken' in w2v_model.wv:
    similar_words = w2v_model.wv.most_similar('chicken', topn=10)
    print("\nWords similar to 'chicken':")
    for word, score in similar_words:
        print(f"{word}: {score:.4f}")

Method 5: Word2Vec with TF-IDF Weighting

Pros:

    Combines semantic understanding with term importance
    Better than simple averaging
    Gives more weight to distinctive terms
    Still dense representation

Cons:

    More complex to implement
    Computationally expensive
    Requires careful tuning
    Two models to maintain

# Create TF-IDF weights dictionary
tfidf_simple = TfidfVectorizer(max_features=10000)
tfidf_matrix = tfidf_simple.fit_transform(df['description_clean'])
feature_names = tfidf_simple.get_feature_names_out()

# Create weighted Word2Vec vectors
X_w2v_tfidf = np.zeros((len(df), 100))

for i, tokens in enumerate(tokenized_descriptions):
    weighted_vectors = []
    total_weight = 0
    
    for word in tokens:
        if word in w2v_model.wv and word in feature_names:
            # Get TF-IDF weight
            word_idx = np.where(feature_names == word)[0]
            if len(word_idx) > 0:
                tfidf_weight = tfidf_matrix[i, word_idx[0]]
                weighted_vectors.append(w2v_model.wv[word] * tfidf_weight)
                total_weight += tfidf_weight
    
    if weighted_vectors and total_weight > 0:
        X_w2v_tfidf[i] = np.sum(weighted_vectors, axis=0) / total_weight
    else:
        X_w2v_tfidf[i] = np.zeros(100)

print(f"TF-IDF weighted Word2Vec shape: {X_w2v_tfidf.shape}")

Method 6: Keras Tokenizer with Embeddings

Pros:

    Integrates directly with neural networks
    Learns embeddings during training
    Can capture task-specific patterns
    Flexible architecture

Cons:

    Requires neural network training
    Needs more data
    Longer training time
    Less interpretable

# Initialize Keras Tokenizer
keras_tokenizer = Tokenizer(
    num_words=10000,
    oov_token='<OOV>',
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)

# Fit on texts
keras_tokenizer.fit_on_texts(df['description_clean'])

# Convert to sequences
sequences = keras_tokenizer.texts_to_sequences(df['description_clean'])

# Pad sequences
max_length = 100  # Maximum sequence length
X_padded = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

print(f"Keras sequences shape: {X_padded.shape}")
print(f"Vocabulary size: {len(keras_tokenizer.word_index)}")
print(f"Sample sequence: {X_padded[0][:20]}")

# Get word index
word_index = keras_tokenizer.word_index
print(f"Total unique tokens: {len(word_index)}")

Neural Network Architectures for Classification
Architecture 1: Autoencoder for Clustering (Unsupervised)

Use Case: Learn compressed representations and cluster in latent space

Pros:

    Unsupervised dimensionality reduction
    Can discover hidden patterns
    Good for unlabeled data

Cons:

    May not align with classification goals
    Requires post-processing for clusters

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.cluster import KMeans

# Prepare data (using TF-IDF as example)
X_dense = X_tfidf.toarray()
scaler = StandardScaler(with_mean=False)
X_scaled = scaler.fit_transform(X_dense)

# Define autoencoder architecture
input_dim = X_scaled.shape[1]
encoding_dim = 128

input_layer = Input(shape=(input_dim,))
encoded = Dense(512, activation='relu')(input_layer)
encoded = Dropout(0.3)(encoded)
encoded = Dense(256, activation='relu')(encoded)
encoded = Dropout(0.3)(encoded)
encoded = Dense(encoding_dim, activation='relu', name='encoded_layer')(encoded)

decoded = Dense(256, activation='relu')(encoded)
decoded = Dropout(0.3)(decoded)
decoded = Dense(512, activation='relu')(decoded)
decoded = Dropout(0.3)(decoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

# Compile autoencoder
autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print(autoencoder.summary())

# Train autoencoder
history_ae = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

# Extract encoder
encoder = Model(input_layer, encoded)
encoded_features = encoder.predict(X_scaled)

print(f"Encoded features shape: {encoded_features.shape}")

# Apply K-Means clustering on encoded features
n_clusters = 10  # Adjust based on expected food categories
kmeans_ae = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters_ae = kmeans_ae.fit_predict(encoded_features)

df['cluster_autoencoder'] = clusters_ae
print(f"\nCluster distribution:\n{pd.Series(clusters_ae).value_counts().sort_index()}")

Architecture 2: Deep Embedding Clustering (DEC)

Use Case: Simultaneously learn features and cluster assignments

Pros:

    End-to-end clustering
    Learns task-specific representations
    Better than sequential approach

Cons:

    More complex training
    Requires initialization
    Computationally intensive

from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K

# First, pretrain autoencoder (use the one from above)
# Then use encoded features for clustering

# Initialize cluster centers using K-Means
kmeans_init = KMeans(n_clusters=n_clusters, n_init=20, random_state=42)
initial_clusters = kmeans_init.fit_predict(encoded_features)
cluster_centers = kmeans_init.cluster_centers_

print(f"Initial cluster centers shape: {cluster_centers.shape}")

# Calculate soft assignments using Student's t-distribution
alpha = 1.0  # Degrees of freedom

# Calculate Q (soft assignments)
q_numerator = 1.0 / (1.0 + (np.sum(np.square(
    encoded_features[:, np.newaxis, :] - cluster_centers[np.newaxis, :, :]
), axis=2) / alpha))

q = q_numerator ** ((alpha + 1.0) / 2.0)
q = q / np.sum(q, axis=1, keepdims=True)

print(f"Soft assignments shape: {q.shape}")

# Calculate target distribution P
weight = q ** 2 / np.sum(q, axis=0)
p = (weight.T / np.sum(weight, axis=1)).T

print(f"Target distribution shape: {p.shape}")

# Build DEC model
input_dec = Input(shape=(input_dim,))
encoded_dec = encoder(input_dec)

# Add clustering layer output
# For simplicity, we'll use the encoder and refine with additional training
dec_model = Model(input_dec, encoded_dec)

# Fine-tune with clustering objective (simplified version)
# In practice, you'd implement a custom clustering layer
dec_features = dec_model.predict(X_scaled)

kmeans_dec = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters_dec = kmeans_dec.fit_predict(dec_features)

df['cluster_dec'] = clusters_dec
print(f"\nDEC Cluster distribution:\n{pd.Series(clusters_dec).value_counts().sort_index()}")

Architecture 3: Variational Autoencoder (VAE) for Clustering

Use Case: Probabilistic clustering with generated representations

Pros:

    Probabilistic framework
    Smooth latent space
    Can generate new samples
    Better regularization

Cons:

    More complex to train
    Requires careful tuning of KL weight
    May sacrifice reconstruction quality

from tensorflow.keras.layers import Lambda
from tensorflow.keras.losses import binary_crossentropy

# VAE parameters
latent_dim = 32
intermediate_dim = 256

# Encoder
vae_input = Input(shape=(input_dim,))
h = Dense(intermediate_dim, activation='relu')(vae_input)
h = Dropout(0.3)(h)

z_mean = Dense(latent_dim, name='z_mean')(h)
z_log_var = Dense(latent_dim, name='z_log_var')(h)

# Sampling layer
epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim), mean=0., stddev=1.0)
z = z_mean + K.exp(z_log_var / 2) * epsilon

# Decoder
decoder_h = Dense(intermediate_dim, activation='relu')
decoder_mean = Dense(input_dim, activation='sigmoid')

h_decoded = decoder_h(z)
